# Daily Challenge: Text Analysis of books using word clouds

**What you'll learn:** text preprocessing · text analysis · Bag of Words (BoW) · TF-IDF

**Corpus:** three books by Lewis Carroll from Project Gutenberg:
- *Alice's Adventures in Wonderland* (pg11)
- *Through the Looking-Glass, and What Alice Found There* (pg12)
- *A Tangled Tale* (pg29042)

> **Environment:** locally this course should be run in a dedicated virtual environment, for example `python -m venv nlp-env && source nlp-env/bin/activate && pip install nltk spacy wordcloud scikit-learn matplotlib requests`. On Colab the cell below installs what is missing.

In [ ]:
%pip install --quiet nltk spacy wordcloud scikit-learn matplotlib requests

In [ ]:
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import spacy
from collections import Counter

for res in ["punkt", "punkt_tab", "stopwords", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
            "maxent_ne_chunker", "maxent_ne_chunker_tab", "words"]:
    nltk.download(res, quiet=True)

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("nltk", nltk.__version__, "| spaCy", spacy.__version__)

# Part 1 · Text preprocessing

### 1. `load_texts()`: download the books and clean non-words
The function downloads each URL with `requests`, then uses regular expressions to:
- normalise typographic apostrophes (`’` → `'`) so that `Alice’s` stays one word;
- replace every character that is not a letter, an apostrophe or a line break with a space (digits, punctuation, `*`, `_`, `“ ”`, `—` …);
- drop apostrophes that are not inside a word (quotation marks such as `'Off with her head!'`);
- collapse repeated spaces.

Line breaks are kept on purpose, so that the Gutenberg header and footer lines can still be located in step 2.

In [ ]:
URLS = {
    "Alice's Adventures in Wonderland": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "Through the Looking-Glass": "https://www.gutenberg.org/cache/epub/12/pg12.txt",
    "A Tangled Tale": "https://www.gutenberg.org/cache/epub/29042/pg29042.txt",
}
TITLES = list(URLS)

def clean_text(text):
    text = text.replace("\ufeff", "").replace("’", "'").replace("‘", "'")
    text = re.sub(r"[^A-Za-z'\n]", " ", text)              # keep only letters, apostrophes and newlines
    text = re.sub(r"(?<![A-Za-z])'|'(?![A-Za-z])", " ", text)  # apostrophes that are quotation marks
    text = re.sub(r"[ \t]+", " ", text)                      # collapse spaces
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()

def load_texts(urls):
    corpus = []
    for url in urls:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        response.encoding = "utf-8"
        corpus.append(clean_text(response.text))
    return corpus

corpus = load_texts(URLS.values())
for title, text in zip(TITLES, corpus):
    print(f"{title:35s}: {len(text):7d} characters")

### 2. First 200 characters of each text

In [ ]:
for title, text in zip(TITLES, corpus):
    print(f"--- {title}\n{text[:200]!r}\n")

**Yes, there are irrelevant parts.** Each file starts with the Project Gutenberg header ("The Project Gutenberg eBook of …", "This ebook is for the use of anyone anywhere…", release date, language, credits) and ends with about 350 lines of licence text (the "Project Gutenberg License", donation info…). These words (*Project*, *Gutenberg*, *electronic*, *work*, *license*, *foundation*…) would pollute the word counts; the example word cloud of the exercise even shows "Project Gutenberg" and "electronic work".

We keep only the text between the line containing ` START` (`*** START OF THE PROJECT GUTENBERG EBOOK … ***`) and the line containing ` END` (`*** END OF THE PROJECT GUTENBERG EBOOK … ***`). The stars were removed by the cleaning, so we search for the phrases `START OF THE PROJECT GUTENBERG EBOOK` and `END OF THE PROJECT GUTENBERG EBOOK`. *A Tangled Tale* also has a producer credit and a transcriber's note right after the START marker, which we remove too.

In [ ]:
START_MARK = "START OF THE PROJECT GUTENBERG EBOOK"
END_MARK = "END OF THE PROJECT GUTENBERG EBOOK"

def trim_gutenberg(text):
    start = text.find(START_MARK)
    start = text.find("\n", start) + 1          # skip the rest of the START line (the title)
    end = text.find(END_MARK)
    body = text[start:end].strip()
    # A Tangled Tale: drop the producer credit + transcriber's note before the actual title page
    if "Transcriber's note" in body[:1000]:
        body = body[body.find("A TANGLED TALE"):]
    return body

books = [trim_gutenberg(t) for t in corpus]
for title, raw, book in zip(TITLES, corpus, books):
    print(f"--- {title}: {len(raw)} -> {len(book)} characters")
    print(repr(book[:200]))
    print("... ends with:", repr(book[-120:]), "\n")

### 3. Tokenization: first 150 tokens of each book

In [ ]:
from nltk.tokenize import word_tokenize

tokens = [word_tokenize(book) for book in books]
for title, toks in zip(TITLES, tokens):
    print(f"--- {title}: {len(toks)} tokens")
    print(toks[:150], "\n")

### 4. Remove stopwords (NLTK)
We lowercase the tokens first (the stopword list is lowercase, so `I` and `The` would otherwise survive). NLTK's word tokenizer also splits contractions into pieces such as `n't`, `'s`, `'ll`, `'m`, `'ve`. These pieces are not in the stopword list, so we drop them as well, together with one-letter tokens.

In [ ]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

def remove_stopwords(toks):
    return [t for t in (w.lower() for w in toks)
            if t not in stop_words and "'" not in t and len(t) > 1]

tokens_lower = [[t.lower() for t in toks] for toks in tokens]
tokens_nostop = [remove_stopwords(toks) for toks in tokens]

check = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "the", "and", "she", "it"]
rows = []
for title, before, after in zip(TITLES, tokens_lower, tokens_nostop):
    rows.append({"book": title, "tokens before": len(before), "tokens after": len(after),
                 **{f"'{w}' before/after": f"{before.count(w)} / {after.count(w)}" for w in check}})
pd.DataFrame(rows).set_index("book").T

### 5. Stemming: `PorterStemmer`, first 50 stemmed tokens

In [ ]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

stemmed = [[stemmer.stem(t) for t in toks] for toks in tokens_nostop]
for title, toks, st in zip(TITLES, tokens_nostop, stemmed):
    print(f"--- {title}")
    print(st[:50], "\n")

### 6. Lemmatization with spaCy `en_core_web_sm`
The lemma is available as the `token.lemma_` attribute. spaCy's lemmatizer depends on the POS tagger, so we run the pipeline on the **trimmed text with its original casing and sentence structure**. Then we apply the same filters (lowercase, no stopwords, letters only) so that the result can be compared with the stems. The parser and NER components are disabled for speed.

In [ ]:
nlp.max_length = 2_000_000
lemmatized = []
for title, book in zip(TITLES, books):
    doc = nlp(book.replace("\n", " "), disable=["parser", "ner"])
    lemmas = [tok.lemma_.lower() for tok in doc
              if tok.is_alpha and tok.lemma_.lower() not in stop_words and len(tok.lemma_) > 1]
    lemmatized.append(lemmas)
    print(f"--- {title}: {len(lemmas)} lemmas")
    print(lemmas[:50], "\n")

### 7. Stemmed vs lemmatized tokens

In [ ]:
# Side-by-side on the same words: stem vs spaCy lemma (computed in context)
pairs = Counter()
for book in books[:1]:
    doc = nlp(book.replace("\n", " "), disable=["parser", "ner"])
    for tok in doc:
        w = tok.text.lower()
        if tok.is_alpha and w not in stop_words and len(w) > 1:
            pairs[(w, stemmer.stem(w), tok.lemma_.lower())] += 1

examples = ["was", "went", "said", "thought", "began", "looked", "rabbit", "curious", "curiously", "very",
            "feet", "mice", "better", "happens", "pictures", "conversations", "nothing", "daisies", "queen",
            "anxiously", "beautifully", "remarked", "having", "tried", "considering", "trying", "eyes", "alice"]
rows = []
for (w, s, l), n in pairs.items():
    if w in examples:
        rows.append({"word": w, "Porter stem": s, "spaCy lemma": l, "count": n})
cmp_df = (pd.DataFrame(rows).sort_values(["word", "count"], ascending=[True, False])
            .drop_duplicates("word").set_index("word"))
display(cmp_df)

distinct = lambda seqs: len(set(t for s in seqs for t in s))
print("Distinct words  (no stopwords):", distinct(tokens_nostop))
print("Distinct stems                :", distinct(stemmed))
print("Distinct lemmas               :", distinct(lemmatized))
real_words = set(w for s in tokens_nostop for w in s)
non_words = [s for s in set(t for st in stemmed for t in st) if s not in real_words]
print(f"Stems that are not real words : {len(non_words)} e.g. {sorted(non_words)[:15]}")

ANALYSIS_STEM

### 8. POS tags of each text (NLTK)

In [ ]:
from nltk import pos_tag

pos_tags = [pos_tag(toks) for toks in tokens]        # original casing, all tokens -> better context
for title, tags in zip(TITLES, pos_tags):
    print(f"--- {title}: first 25 tags")
    print(tags[:25], "\n")

pos_counts = pd.DataFrame({title: Counter(t for _, t in tags) for title, tags in zip(TITLES, pos_tags)}).fillna(0).astype(int)
pos_share = (pos_counts / pos_counts.sum() * 100).round(1)
display(pos_share.loc[pos_counts.sum(axis=1).sort_values(ascending=False).index[:12]])

### 9. Named entities of each text (NLTK `ne_chunk`)
`ne_chunk` needs POS tags and proper capitalisation, and it works sentence by sentence. Because of this we use the trimmed text **before** removing punctuation: same START/END trimming, but without the non-word cleaning step. That way `sent_tokenize` can split the sentences.

In [ ]:
from nltk import sent_tokenize, ne_chunk
from nltk.tree import Tree

raw_books = []
for url in URLS.values():
    r = requests.get(url, timeout=60); r.encoding = "utf-8"
    t = r.text.replace("\ufeff", "").replace("’", "'").replace("‘", "'")
    s = t.find("\n", t.find(START_MARK)) + 1
    e = t.rfind("\n", 0, t.find(END_MARK))
    raw_books.append(t[s:e])

def extract_entities(text):
    entities = []
    for sent in sent_tokenize(text):
        tree = ne_chunk(pos_tag(word_tokenize(sent)))
        for node in tree:
            if isinstance(node, Tree):
                entities.append((" ".join(w for w, _ in node.leaves()), node.label()))
    return entities

entities = [extract_entities(t) for t in raw_books]
for title, ents in zip(TITLES, entities):
    c = Counter(ents)
    print(f"--- {title}: {len(ents)} entity mentions, {len(c)} distinct")
    print("labels:", dict(Counter(lbl for _, lbl in ents)))
    print("top 15:", c.most_common(15), "\n")

ANALYSIS_NER

# Part 2 · Analysing the text

### 1. Word cloud of each book
We use the **spaCy lemmas without stopwords**, so that *say/said/says* or *go/went/goes* count as one word, as in the example in the exercise.

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for ax, title, lemmas in zip(axes, TITLES, lemmatized):
    wc = WordCloud(width=800, height=800, background_color="white", colormap="viridis",
                   random_state=42, collocations=False).generate(" ".join(lemmas))
    ax.imshow(wc, interpolation="bilinear"); ax.axis("off"); ax.set_title(title, fontsize=16)
plt.tight_layout(); plt.show()

### 2. Bag of Words: the five most frequent words across all books
**Which text should we use?** Not the raw text: it would be dominated by stopwords (*the, and, to, she…*) and inflections would split counts (*said/say/says*). Stems merge inflections but give non-words that are hard to read (*alic, littl, queen*). The best input is the **lemmatized text without stopwords**: real dictionary words with the inflected forms merged.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = [" ".join(l) for l in lemmatized]          # one document per book
count_vec = CountVectorizer()
bow = count_vec.fit_transform(docs)
vocab = count_vec.get_feature_names_out()
print("BoW matrix shape (documents x vocabulary):", bow.shape)

total = np.asarray(bow.sum(axis=0)).ravel()
top5_idx = total.argsort()[::-1][:5]
top5 = pd.DataFrame({"word": vocab[top5_idx], "vocab index": top5_idx, "total count": total[top5_idx],
                     **{t: bow[i].toarray().ravel()[top5_idx] for i, t in enumerate(TITLES)}})
top5

### 3. Print the BoW and read the numbers

In [ ]:
print("Printed BoW (first 40 vocabulary columns of each document):")
print(bow[:, :40])        # scikit-learn sparse format: (document, word index)  count
print()
for word in ["alice", "say", "queen", "king", "knight", "clara"]:
    idx = count_vec.vocabulary_[word]
    print(f"'{word}' has index {idx:5d} -> counts per document {bow[:, idx].toarray().ravel().tolist()}")

**How to read the BoW printout `(d, i)  n`:**
- **d = document number**: 0 = *Alice's Adventures in Wonderland*, 1 = *Through the Looking-Glass*, 2 = *A Tangled Tale* (the order of `docs`).
- **i = index of the word** in the vocabulary (`count_vec.vocabulary_` / `get_feature_names_out()`, sorted alphabetically).
- **n = how many times** that word appears in that document.

Only non-zero entries are stored (sparse matrix). For example the line for `say` in document 0 shows how often *say/said/says* appears in *Alice in Wonderland*.

### 4. Pie plot of the 5 most frequent words

In [ ]:
def pie_top_words(words, values, title, ax=None, fmt="{:d}"):
    ax = ax or plt.gca()
    labels = [f"{w} ({fmt.format(v)})" for w, v in zip(words, values)]
    colors = plt.cm.Greens(np.linspace(0.85, 0.4, len(words)))
    ax.pie(values, labels=labels, autopct="%1.1f%%", startangle=140, colors=colors,
           textprops={"fontsize": 11}, wedgeprops={"edgecolor": "white"})
    ax.set_title(title, fontsize=13)

plt.figure(figsize=(7, 7))
pie_top_words(top5["word"], top5["total count"].astype(int), "Top 5 most frequent words (all books, BoW)")
plt.show()

In [ ]:
# The same per book, for comparison with TF-IDF later
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for d, (ax, title) in enumerate(zip(axes, TITLES)):
    row = bow[d].toarray().ravel()
    idx = row.argsort()[::-1][:5]
    pie_top_words(vocab[idx], row[idx].astype(int), f"BoW top 5: {title}", ax)
plt.tight_layout(); plt.show()

ANALYSIS_BOW

# Part 3 · Solving the frequency problem using TF-IDF

### 1. TF-IDF BoW
`min_df=1, max_df=2`: with only 3 documents, an integer `max_df=2` means "**ignore words that appear in more than 2 documents**", so words present in all three books (*say, go, little, think, time…*) are removed. The remaining words get an IDF weight that is higher when they appear in only one book.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer(min_df=1, max_df=2)
tfidf = tfidf_vec.fit_transform(docs)
tfidf_vocab = tfidf_vec.get_feature_names_out()
print("TF-IDF matrix shape:", tfidf.shape, "(vs", bow.shape, "for the plain BoW)")
print(f"Words removed because they appear in all 3 books: {bow.shape[1] - tfidf.shape[1]}")
removed = sorted(set(vocab) - set(tfidf_vocab), key=lambda w: -total[count_vec.vocabulary_[w]])
print("Most frequent removed words:", removed[:20])
print()
print(tfidf[:1, :40])

In [ ]:
tfidf_top = {}
for d, title in enumerate(TITLES):
    row = tfidf[d].toarray().ravel()
    idx = row.argsort()[::-1][:5]
    tfidf_top[title] = list(zip(tfidf_vocab[idx], row[idx].round(3)))
pd.DataFrame({t: [f"{w} ({s})" for w, s in v] for t, v in tfidf_top.items()}, index=range(1, 6))

### 2. Pie plots of the 5 most relevant words per document

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for ax, (title, items) in zip(axes, tfidf_top.items()):
    words, scores = zip(*items)
    pie_top_words(words, scores, f"TF-IDF top 5: {title}", ax, fmt="{:.3f}")
plt.tight_layout(); plt.show()

In [ ]:
# Bonus: TF-IDF word clouds (weights instead of raw counts)
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for d, (ax, title) in enumerate(zip(axes, TITLES)):
    row = tfidf[d].toarray().ravel()
    weights = {w: s for w, s in zip(tfidf_vocab, row) if s > 0}
    wc = WordCloud(width=800, height=800, background_color="white", random_state=42).generate_from_frequencies(weights)
    ax.imshow(wc, interpolation="bilinear"); ax.axis("off"); ax.set_title(f"TF-IDF: {title}", fontsize=16)
plt.tight_layout(); plt.show()

ANALYSIS_TFIDF